# 全功能示例（可在 Jupyter 里直接运行）

这个 notebook 展示如何把本仓库当作“库”来调用：
- surface 检测
- site primitives
- pose 采样
- basin（异常过滤 + 能窗 + 去重）
- node 输出
- conformer_md（可选）

建议先在仓库根目录执行：

```bash
pip install -e .
```


In [ ]:
from pathlib import Path
import os
import json
import numpy as np

from ase.build import fcc211, molecule
from ase.io import write

from adsorption_ensemble.surface import SurfacePreprocessor, export_surface_detection_report
from adsorption_ensemble.site import PrimitiveBuilder
from adsorption_ensemble.pose import PoseSampler, PoseSamplerConfig
from adsorption_ensemble.basin import BasinBuilder, BasinConfig
from adsorption_ensemble.node import NodeConfig, basin_to_node

out_root = Path("artifacts/notebook_full_usage")
out_root.mkdir(parents=True, exist_ok=True)

slab = fcc211("Pt", size=(6, 4, 4), vacuum=12.0)
ads = molecule("CO")

out_root.as_posix()

In [ ]:
pre = SurfacePreprocessor(min_surface_atoms=6)
ctx = pre.build_context(slab)
surface_files = export_surface_detection_report(slab, ctx, out_root / "surface_report")
surface_files

In [ ]:
primitives = PrimitiveBuilder().build(slab, ctx)
len(primitives)

In [ ]:
sampler = PoseSampler(
    PoseSamplerConfig(
        n_rotations=2,
        n_azimuth=6,
        n_shifts=1,
        shift_radius=0.0,
        min_height=1.6,
        max_height=2.6,
        height_step=0.2,
        random_seed=0,
        max_poses_per_site=4,
    )
)
poses = sampler.sample(
    slab=slab,
    adsorbate=ads,
    primitives=primitives[:4],
    surface_atom_ids=ctx.detection.surface_atom_ids,
)
pose_frames = [slab + p.atoms for p in poses]
write((out_root / "pose_pool.extxyz").as_posix(), pose_frames)
len(pose_frames)

## Basin（异常过滤 + 能窗 + 去重）与 Node 输出

默认使用 `mace_node_l2` 做 basin 去重；descriptor 比较使用 adsorbate-only 的逐原子 MACE 特征 L2。若你只是做无 MACE 的 smoke run，可显式改成 `binding_surface_distance`。

In [ ]:
cfg = BasinConfig(
    relax_maxf=0.10,
    relax_steps=2,
    energy_window_ev=1.0,
    dedup_metric="rmsd",
    rmsd_threshold=0.10,
    binding_tau=1.15,
    desorption_min_bonds=0,
    work_dir=out_root / "basin_work",
)
basin_out = BasinBuilder(config=cfg).build(
    frames=pose_frames,
    slab_ref=slab,
    adsorbate_ref=ads,
    slab_n=len(slab),
    normal_axis=int(ctx.classification.normal_axis),
)

basins_frames = []
for b in basin_out.basins:
    a = b.atoms.copy()
    a.info["basin_id"] = int(b.basin_id)
    a.info["energy_ev"] = float(b.energy_ev)
    a.info["signature"] = str(b.signature)
    basins_frames.append(a)
write((out_root / "basins.extxyz").as_posix(), basins_frames)

energy_min = basin_out.summary.get("energy_min_ev", None)
try:
    energy_min_ev = None if energy_min is None else float(energy_min)
except Exception:
    energy_min_ev = None
ncfg = NodeConfig(bond_tau=1.20, node_hash_len=20)
nodes = [basin_to_node(b, slab_n=len(slab), cfg=ncfg, energy_min_ev=energy_min_ev) for b in basin_out.basins]
(out_root / "nodes.json").write_text(
    json.dumps(
        [
            {
                "node_id": str(n.node_id),
                "basin_id": int(n.basin_id),
                "canonical_order": [int(x) for x in n.canonical_order],
                "atomic_numbers": [int(x) for x in n.atomic_numbers],
                "internal_bonds": [(int(i), int(j)) for i, j in n.internal_bonds],
                "binding_pairs": [(int(i), int(j)) for i, j in n.binding_pairs],
                "denticity": int(n.denticity),
                "relative_energy_ev": (None if n.relative_energy_ev is None else float(n.relative_energy_ev)),
                "provenance": dict(n.provenance),
            }
            for n in nodes
        ],
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

{
    "n_poses": len(pose_frames),
    "n_basins": len(basin_out.basins),
    "n_nodes": len(nodes),
    "out_root": out_root.as_posix(),
}

## Conformer-MD（可选）

如果你有 `xtb` 可执行文件，可以用 `ConformerMDSampler` 跑真实的 MD；否则可以先用测试里的 Fake runner 把流程跑通。

In [ ]:
import shutil
from adsorption_ensemble.conformer_md import ConformerMDSampler, ConformerMDSamplerConfig, GeometryPairDistanceDescriptor

cfg2 = ConformerMDSamplerConfig()
cfg2.output.work_dir = out_root / "conformer_md"
cfg2.selection.preselect_k = 10
cfg2.selection.mode = "fps"
cfg2.selection.energy_window_ev = 0.20
cfg2.selection.rmsd_threshold = 0.02

xtb_path = shutil.which("xtb")
if xtb_path is None:
    from tools.full_repo_example import FakeMDRunner, FakeRelaxBackend
    sampler2 = ConformerMDSampler(
        config=cfg2,
        md_runner=FakeMDRunner(n_frames=60),
        descriptor_extractor=GeometryPairDistanceDescriptor(),
        relax_backend=FakeRelaxBackend(),
    )
else:
    sampler2 = ConformerMDSampler(config=cfg2, descriptor_extractor=GeometryPairDistanceDescriptor())

result = sampler2.run(molecule("H2O"), job_name="example")
{
    "xtb": (None if xtb_path is None else xtb_path),
    "n_conformers": len(result.conformers),
    "out_dir": (cfg2.output.work_dir / "example").as_posix(),
}